In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries

from darts.utils.missing_values import missing_values_ratio, fill_missing_values
from darts.metrics.metrics import mape, mae, rmse, mase

from ts_missing_values.gap_filling import *
from ts_missing_values.comparison import *
from ts_missing_values.preprocessing import *
from ts_missing_values.utility import *

In [ ]:
sensors = pd.read_csv('./data/synthetic_traffic.csv')
sensors.timestamp = pd.to_datetime(sensors.timestamp)
sensors = sensors.set_index('timestamp')
sensors

Wrap each sensor into a Darts TimeSeries, plot it for inspection, and store series and names into a dict and list for future use.

In [ ]:
sensors_dict = {}

for i, name in enumerate(sensors.columns):
    ts_original = TimeSeries(times=sensors.index, values=sensors[name].values)
    plt.figure(figsize=(15,4))
    ts_original.plot()
    
    plt.title(f'{i} - {name}')
    sensors_dict[name] = ts_original

sensors_list = list(sensors_dict.values())
sensors_names = list(sensors_dict.keys())

Change series_index to try on different sensors.

Here you can see the original series, the density of missing values (and hig density highlighed) and the series transformed.

In [ ]:
series_index = 20

ts_original = sensors_list[series_index]
candidates_list = sensors_list.copy()
candidates_list.remove(ts_original)

window = 168
threshold = 0.5
ts_to_fill, high_gap_density_series = gap_transform(ts_original, window_size=window, threshold_percentage=threshold, plot=True, return_density_series=True)
print('Applying gap unification on the same series with window:', window)

Time series filling

In [ ]:
ts_filled = universal_filling(
    series=ts_to_fill,
    candidates=candidates_list,
    # sporadic missing values filling
    num_values=3,
    distance=168,
    # gap filling
    k=5,
    metric="rmse",
    transform="none",
)

plt.figure(figsize=(15,4))
ts_filled.plot(label='filled')
ts_original.plot(label='original')

The series is fully filled

In [ ]:
density = extract_gap_density(ts_filled, plot=True)

Using plotly for closer inspection

In [ ]:
from plotly import graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_filled.time_index, y=ts_filled.univariate_values(), name='filled'))
fig.add_trace(go.Scatter(x=ts_original.time_index, y=ts_original.univariate_values(), name='original'))

Just small gap filling

In [ ]:
ts = sporadic_filling(
    ts_to_fill,high_gap_density_series,
    distance=168,
    num_values=5
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts.time_index, y=ts.univariate_values(), name='small gap filled'))
fig.add_trace(go.Scatter(x=ts_original.time_index, y=ts_original.univariate_values(), name='original'))

Just large gap filling

In [ ]:
ts = gap_filling(
    ts_to_fill,
    high_gap_density_series=high_gap_density_series,
    candidates=candidates_list,
    k=10,
    metric="max_distance",
    transform='diff'
)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts.time_index, y=ts.univariate_values(), name='small gap filled'))
fig.add_trace(go.Scatter(x=ts_original.time_index, y=ts_original.univariate_values(), name='original'))